# Phase 05 — Qdrant indexing

This notebook indexes **only real BAAI/bge-m3 dense embeddings** produced by Phase 04. It uses an in-memory Qdrant instance for a repeatable notebook experiment, configures cosine distance with the vector dimension from the artifact, preserves required chunk metadata as the payload, and checks collection statistics, a metadata filter, and duplicate-free re-indexing.

> **Strict preflight:** No fallback model, mock vector, or fabricated embedding is permitted. If Phase 04 did not generate an embedding artifact, this notebook writes a truthful status record and does not create a collection.

This phase performs collection-management verification only. It does not implement user-query retrieval.

## Environment

When real embeddings are available, install the Qdrant Python client in the notebook environment:

```bash
pip install qdrant-client
```

The implementation uses `QdrantClient(":memory:")`, `VectorParams(size=..., distance=Distance.COSINE)`, deterministic point IDs, `upsert`, `count`, and filtered `scroll`. The approach follows Qdrant’s collection, point, and filtering documentation.

In [1]:
from __future__ import annotations

import json
import uuid
from collections import Counter
from pathlib import Path
from typing import Any

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
EMBEDDING_STATUS_PATH = PROCESSED_DIR / "introduction_to_business_bge_m3_embedding_status.json"
EMBEDDINGS_PATH = PROCESSED_DIR / "introduction_to_business_bge_m3_embeddings.jsonl"
QDRANT_STATUS_PATH = PROCESSED_DIR / "introduction_to_business_qdrant_indexing_status.json"
COLLECTION_NAME = "openstax_introduction_to_business_bge_m3"
REQUIRED_PAYLOAD_FIELDS = ("chunk_id", "text", "source", "page", "chapter", "section")

def write_status(payload: dict[str, Any]) -> None:
    QDRANT_STATUS_PATH.parent.mkdir(parents=True, exist_ok=True)
    QDRANT_STATUS_PATH.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print(f"Project root: {PROJECT_ROOT}")
print(f"Expected embeddings: {EMBEDDINGS_PATH}")

Project root: /home/ubuntu/business-knowledge-ai
Expected embeddings: /home/ubuntu/business-knowledge-ai/data/processed/introduction_to_business_bge_m3_embeddings.jsonl


In [2]:
embedding_status = json.loads(EMBEDDING_STATUS_PATH.read_text(encoding="utf-8"))
embeddings_generated = bool(embedding_status.get("embeddings_generated"))
embedding_file_exists = EMBEDDINGS_PATH.is_file() and EMBEDDINGS_PATH.stat().st_size > 0
PIPELINE_READY = embeddings_generated and embedding_file_exists

preflight = {
    "phase": "05_qdrant_indexing",
    "collection_name": COLLECTION_NAME,
    "embedding_model": embedding_status.get("model_name"),
    "embedding_status": embedding_status.get("status"),
    "embeddings_generated": embeddings_generated,
    "embedding_artifact_exists": embedding_file_exists,
    "no_vector_substitution": True,
    "no_collection_created_without_real_embeddings": True,
}

if not PIPELINE_READY:
    preflight.update({
        "status": "blocked_missing_real_embeddings",
        "limitation": (
            "Phase 04 did not produce a real BAAI/bge-m3 embedding artifact. "
            "No Qdrant collection is created because vectors must not be substituted or fabricated."
        ),
    })
    write_status(preflight)
    print("Qdrant indexing blocked: real BGE-M3 embeddings are unavailable.")
    print(f"Phase 04 status: {embedding_status.get('status')}")
else:
    preflight.update({"status": "ready_for_real_qdrant_indexing"})
    print("Preflight passed: real BGE-M3 embeddings are available for indexing.")

Qdrant indexing blocked: real BGE-M3 embeddings are unavailable.
Phase 04 status: blocked_by_resource_preflight


## Point and payload schema

Each Qdrant point uses a deterministic UUIDv5 derived from `chunk_id`, so re-indexing the same artifact updates existing points rather than introducing duplicates. The payload preserves these required fields exactly: `chunk_id`, `text`, `source`, `page`, `chapter`, and `section`.

The collection dimension is derived from a real vector at runtime and must be 1,024 for BGE-M3 dense embeddings. The collection uses `Distance.COSINE`; the input vectors are already L2-normalized in Phase 04.

In [3]:
if PIPELINE_READY:
    embedding_records = [
        json.loads(line)
        for line in EMBEDDINGS_PATH.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    if not embedding_records:
        raise ValueError("Embedding artifact is empty; indexing is intentionally stopped.")

    missing_payload = {
        field: sum(field not in record for record in embedding_records)
        for field in REQUIRED_PAYLOAD_FIELDS
    }
    if any(missing_payload.values()):
        raise ValueError(f"Missing required payload fields: {missing_payload}")

    def extract_vector(record: dict[str, Any]) -> list[float]:
        vector = record.get("embedding", record.get("dense_embedding"))
        if not isinstance(vector, list) or not vector:
            raise ValueError("Each record must contain a real dense embedding list.")
        return [float(value) for value in vector]

    vector_size = len(extract_vector(embedding_records[0]))
    if vector_size != 1024:
        raise ValueError(f"Expected BGE-M3 dense dimension 1024, received {vector_size}.")
    if any(len(extract_vector(record)) != vector_size for record in embedding_records):
        raise ValueError("Embedding dimensions are inconsistent; indexing is intentionally stopped.")

    print(f"Loaded {len(embedding_records):,} real embedding records.")
    print(f"Vector dimension: {vector_size}")
    print({field: embedding_records[0][field] for field in REQUIRED_PAYLOAD_FIELDS})
else:
    embedding_records: list[dict[str, Any]] = []
    vector_size = None
    print("Artifact-dependent loading skipped.")

Artifact-dependent loading skipped.


In [4]:
if PIPELINE_READY:
    from qdrant_client import QdrantClient, models

    client = QdrantClient(":memory:")
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(size=vector_size, distance=models.Distance.COSINE),
    )

    points = [
        models.PointStruct(
            id=str(uuid.uuid5(uuid.NAMESPACE_URL, record["chunk_id"])),
            vector=extract_vector(record),
            payload={field: record[field] for field in REQUIRED_PAYLOAD_FIELDS},
        )
        for record in embedding_records
    ]
    client.upsert(collection_name=COLLECTION_NAME, points=points, wait=True)
    print(f"Upserted {len(points):,} points into '{COLLECTION_NAME}'.")
else:
    print("Collection creation and upsert skipped: no real embedding artifact.")

Collection creation and upsert skipped: no real embedding artifact.


In [5]:
if PIPELINE_READY:
    collection_info = client.get_collection(COLLECTION_NAME)
    point_count_before_reindex = client.count(COLLECTION_NAME, exact=True).count
    print({
        "collection": COLLECTION_NAME,
        "distance": collection_info.config.params.vectors.distance.value,
        "vector_dimension": collection_info.config.params.vectors.size,
        "point_count": point_count_before_reindex,
    })

    filter_chapter = next((record["chapter"] for record in embedding_records if record["chapter"]), None)
    if filter_chapter is None:
        raise ValueError("No chapter payload value is available for metadata-filter verification.")
    chapter_filter = models.Filter(
        must=[models.FieldCondition(key="chapter", match=models.MatchValue(value=filter_chapter))]
    )
    filtered_points, _ = client.scroll(
        collection_name=COLLECTION_NAME,
        scroll_filter=chapter_filter,
        limit=3,
        with_payload=True,
        with_vectors=False,
    )
    filtered_count = client.count(COLLECTION_NAME, count_filter=chapter_filter, exact=True).count
    assert all(point.payload["chapter"] == filter_chapter for point in filtered_points)
    print({
        "metadata_filter": {"chapter": filter_chapter},
        "matching_points": filtered_count,
        "sample_chunk_ids": [point.payload["chunk_id"] for point in filtered_points],
    })

    client.upsert(collection_name=COLLECTION_NAME, points=points, wait=True)
    point_count_after_reindex = client.count(COLLECTION_NAME, exact=True).count
    assert point_count_after_reindex == point_count_before_reindex == len(points)

    completed_status = {
        **preflight,
        "status": "completed_with_real_bge_m3_embeddings",
        "embedding_count": len(embedding_records),
        "vector_dimension": vector_size,
        "distance": "cosine",
        "metadata_filter": {"field": "chapter", "value": filter_chapter, "match_count": filtered_count},
        "point_count_before_reindex": point_count_before_reindex,
        "point_count_after_reindex": point_count_after_reindex,
        "duplicates_created_by_reindex": point_count_after_reindex != point_count_before_reindex,
        "required_payload_fields": list(REQUIRED_PAYLOAD_FIELDS),
        "retrieval_implemented": False,
    }
    write_status(completed_status)
    print("Re-indexing verified: deterministic point IDs prevented duplicates.")
else:
    print(f"Limitation record saved: {QDRANT_STATUS_PATH}")

Limitation record saved: /home/ubuntu/business-knowledge-ai/data/processed/introduction_to_business_qdrant_indexing_status.json


## Phase boundary

This notebook is limited to Qdrant collection setup and integrity checks. It does **not** embed text, query a collection for user-facing retrieval, perform BM25 or hybrid search, rerank results, call an LLM, or use LangGraph. When Phase 04 successfully produces real BGE-M3 embeddings in an adequately provisioned environment, rerun this notebook to create the in-memory collection and complete the basic verification steps.